In [14]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    GlobalAveragePooling2D
)
import keras_tuner as kt
import pandas as pd

In [79]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [80]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [81]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [82]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [83]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

In [84]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [85]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [86]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [87]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [88]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [89]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [26]:
cnn_bn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    GlobalAveragePooling2D(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [27]:
cnn_bn.compile(
    optimizer="SGD",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
history_bn = cnn_bn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.4088 - loss: 1.7625 - val_accuracy: 0.0533 - val_loss: 2.1641 - learning_rate: 0.0100
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 280s 1s/step - accuracy: 0.4518 - loss: 1.6005 - val_accuracy: 0.1238 - val_loss: 2.1903 - learning_rate: 0.0100
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 290s 1s/step - accuracy: 0.4665 - loss: 1.5136 - val_accuracy: 0.6425 - val_loss: 1.1546 - learning_rate: 0.0100
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 284s 1s/step - accuracy: 0.4817 - loss: 1.4439 - val_accuracy: 0.5093 - val_loss: 1.2710 - learning_rate: 0.0100
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 286s 1s/step - accuracy: 0.5011 - loss: 1.4098 - val_accuracy: 0.3149 - val_loss: 1.5061 - learning_rate: 0.0100
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5171 - loss: 1.3625
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
220/220 ━━━━━━━━━━━━━━━━━━━━ 297s 1s/step - accuracy: 0.5171 - loss: 1.3625 - val_accu

In [29]:
train_loss, lr_train_acc_bn = cnn_bn.evaluate(train_ds)
valid_loss, lr_valid_acc_bn = cnn_bn.evaluate(valid_ds)
test_loss, lr_test_acc_bn = cnn_bn.evaluate(test_ds)
print(lr_train_acc_bn)
print(lr_valid_acc_bn)
print(lr_test_acc_bn)

220/220 ━━━━━━━━━━━━━━━━━━━━ 55s 240ms/step - accuracy: 0.6475 - loss: 1.1276
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 235ms/step - accuracy: 0.6438 - loss: 1.1598
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 235ms/step - accuracy: 0.6401 - loss: 1.1286
0.6475035548210144
0.6438082456588745
0.6400532126426697


In [30]:
bn_results = pd.DataFrame(columns=[
    "Model",
    "Train accuracy",
    "Test accuracy",
    "Valid accuracy"
])
bn_results.loc[len(bn_results)] = [
    "batch normalization using SGD",
    lr_train_acc_bn,
    lr_test_acc_bn,
    lr_valid_acc_bn
]
bn_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,batch normalization using SGD,0.647504,0.640053,0.643808


In [31]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [32]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [33]:
cnn_bn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    GlobalAveragePooling2D(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
cnn_bn.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [35]:
history_bn = cnn_bn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 261s 1s/step - accuracy: 0.4150 - loss: 1.6950 - val_accuracy: 0.0493 - val_loss: 2.5361 - learning_rate: 0.0010
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 258s 1s/step - accuracy: 0.4735 - loss: 1.5073 - val_accuracy: 0.4434 - val_loss: 1.5621 - learning_rate: 0.0010
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 257s 1s/step - accuracy: 0.5039 - loss: 1.4161 - val_accuracy: 0.5313 - val_loss: 1.0780 - learning_rate: 0.0010
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 282s 1s/step - accuracy: 0.5136 - loss: 1.3793 - val_accuracy: 0.5839 - val_loss: 1.0704 - learning_rate: 0.0010
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 288s 1s/step - accuracy: 0.5302 - loss: 1.3391 - val_accuracy: 0.6052 - val_loss: 1.0186 - learning_rate: 0.0010
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 287s 1s/step - accuracy: 0.5245 - loss: 1.3348 - val_accuracy: 0.5047 - val_loss: 1.2309 - learning_rate: 0.0010
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 256s 1s/step - accuracy: 0.5311 - loss: 1.3111 - val_

In [36]:
train_loss, rm_train_acc_bn = cnn_bn.evaluate(train_ds)
valid_loss, rm_valid_acc_bn = cnn_bn.evaluate(valid_ds)
test_loss, rm_test_acc_bn = cnn_bn.evaluate(test_ds)
print(rm_train_acc_bn)
print(rm_valid_acc_bn)
print(rm_test_acc_bn)

220/220 ━━━━━━━━━━━━━━━━━━━━ 60s 262ms/step - accuracy: 0.6110 - loss: 0.9963
47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 257ms/step - accuracy: 0.6079 - loss: 0.9887
47/47 ━━━━━━━━━━━━━━━━━━━━ 942s 20s/step - accuracy: 0.6041 - loss: 1.0034
0.6109843254089355
0.6078562140464783
0.6041250824928284


In [37]:
bn_results.loc[len(bn_results)] = [
    "batch normalization using RMSprop",
    rm_train_acc_bn,
    rm_test_acc_bn,
    rm_valid_acc_bn
]
bn_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,batch normalization using SGD,0.647504,0.640053,0.643808
1,batch normalization using RMSprop,0.610984,0.604125,0.607856


In [38]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [39]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [40]:
cnn_bn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    GlobalAveragePooling2D(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [41]:
cnn_bn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [42]:
history_bn = cnn_bn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 594s 3s/step - accuracy: 0.3989 - loss: 1.7141 - val_accuracy: 0.0439 - val_loss: 3.0802 - learning_rate: 0.0010
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 271s 1s/step - accuracy: 0.4669 - loss: 1.4760 - val_accuracy: 0.2277 - val_loss: 1.9329 - learning_rate: 0.0010
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 262s 1s/step - accuracy: 0.4585 - loss: 1.5040 - val_accuracy: 0.2350 - val_loss: 1.8125 - learning_rate: 0.0010
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 273s 1s/step - accuracy: 0.4849 - loss: 1.3957 - val_accuracy: 0.4294 - val_loss: 1.4360 - learning_rate: 0.0010
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 268s 1s/step - accuracy: 0.5111 - loss: 1.3462 - val_accuracy: 0.2164 - val_loss: 1.7753 - learning_rate: 0.0010
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 270s 1s/step - accuracy: 0.5294 - loss: 1.3099 - val_accuracy: 0.3356 - val_loss: 1.7638 - learning_rate: 0.0010
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 263s 1s/step - accuracy: 0.5355 - loss: 1.

In [43]:
train_loss, adam_train_acc_bn = cnn_bn.evaluate(train_ds)
valid_loss, adam_valid_acc_bn = cnn_bn.evaluate(valid_ds)
test_loss, adam_test_acc_bn = cnn_bn.evaluate(test_ds)
print(adam_train_acc_bn)
print(adam_valid_acc_bn)
print(adam_test_acc_bn)

220/220 ━━━━━━━━━━━━━━━━━━━━ 58s 251ms/step - accuracy: 0.5581 - loss: 1.2155
47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 244ms/step - accuracy: 0.5519 - loss: 1.2439
47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 249ms/step - accuracy: 0.5429 - loss: 1.2715
0.5580599308013916
0.5519307851791382
0.5429141521453857


In [45]:
bn_results.loc[len(bn_results)] = [
    "batch normalization using adam",
    adam_train_acc_bn,
    adam_test_acc_bn,
    adam_valid_acc_bn
]
bn_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,batch normalization using SGD,0.647504,0.640053,0.643808
1,batch normalization using RMSprop,0.610984,0.604125,0.607856
2,batch normalization using adam,0.558060,0.542914,0.551931


In [46]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 16

In [47]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [48]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [49]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [50]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [51]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [52]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [53]:
cnn_bn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    GlobalAveragePooling2D(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [54]:
cnn_bn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [55]:
batch_size = 16
history_bn = cnn_bn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    batch_size = batch_size
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


439/439 ━━━━━━━━━━━━━━━━━━━━ 265s 594ms/step - accuracy: 0.3853 - loss: 1.7801 - val_accuracy: 0.1292 - val_loss: 2.3154
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 250s 565ms/step - accuracy: 0.4324 - loss: 1.5662 - val_accuracy: 0.2051 - val_loss: 1.9187
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 265s 598ms/step - accuracy: 0.4499 - loss: 1.4709 - val_accuracy: 0.4148 - val_loss: 1.4904
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 271s 611ms/step - accuracy: 0.4693 - loss: 1.4096 - val_accuracy: 0.3708 - val_loss: 1.7260
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 306s 688ms/step - accuracy: 0.4760 - loss: 1.4083 - val_accuracy: 0.4547 - val_loss: 1.3994
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 295s 665ms/step - accuracy: 0.4839 - loss: 1.3594 - val_accuracy: 0.2557 - val_loss: 1.8436
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 297s 671ms/step - accuracy: 0.5034 - loss: 1.3285 - val_accuracy: 0.3103 - val_loss: 3.4004
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 319s 663ms/step - accuracy: 0.5016 - loss: 1.31

In [56]:
train_loss, bt16_train_acc_bn = cnn_bn.evaluate(train_ds)
valid_loss, bt16_valid_acc_bn = cnn_bn.evaluate(valid_ds)
test_loss, bt16_test_acc_bn = cnn_bn.evaluate(test_ds)
print(bt16_train_acc_bn)
print(bt16_valid_acc_bn)
print(bt16_test_acc_bn)

439/439 ━━━━━━━━━━━━━━━━━━━━ 66s 144ms/step - accuracy: 0.5127 - loss: 1.3347
94/94 ━━━━━━━━━━━━━━━━━━━━ 13s 134ms/step - accuracy: 0.5180 - loss: 1.3492
94/94 ━━━━━━━━━━━━━━━━━━━━ 12s 130ms/step - accuracy: 0.5023 - loss: 1.3228
0.5126961469650269
0.5179760456085205
0.5023286938667297


In [57]:
bn_results.loc[len(bn_results)] = [
    "batch normalization using batchsize 16",
    bt16_train_acc_bn,
    bt16_test_acc_bn,
    bt16_valid_acc_bn
]
bn_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,batch normalization using SGD,0.647504,0.640053,0.643808
1,batch normalization using RMSprop,0.610984,0.604125,0.607856
2,batch normalization using adam,0.558060,0.542914,0.551931
3,batch normalization using batchsize 16,0.512696,0.502329,0.517976


In [90]:
def build_bn_cnn(hp):
    model=tf.keras.Sequential([
        Conv2D(
            32,
            (3,3),
            activation="relu",
            input_shape=INPUT_SHAPE
        ),
        BatchNormalization(),
        MaxPooling2D(),
        Conv2D(
            64,
            (3,3),
            activation="relu"
        ),
        BatchNormalization(),
        MaxPooling2D(),
        Flatten(),
        Dense(
            hp.Int(
                "units",
                64,
                128,
                64
            ),
            activation="relu"
        ),
        Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])
    model.compile(
        optimizer=hp.Choice(
            "optimizer",
            ["adam","rmsprop"]
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [91]:
bn_tuner = kt.RandomSearch(
    build_bn_cnn,
    objective="val_accuracy",
    max_trials=3,
    directory="tuning",
    project_name="cnn_bn"
)
bn_tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights
)

Trial 3 Complete [00h 17m 24s]
val_accuracy: 0.09454061090946198

Best val_accuracy So Far: 0.32423436641693115
Total elapsed time: 00h 47m 59s


In [92]:
best_bn_cnn = bn_tuner.get_best_models(1)[0]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [93]:
best_hps = bn_tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'units': 64, 'optimizer': 'rmsprop'}


In [94]:
cnn_bn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    GlobalAveragePooling2D(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [95]:
cnn_bn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [98]:
history = cnn_bn.fit(
    train_ds,
    validation_data=valid_ds, 
    epochs=5
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 282s 1s/step - accuracy: 0.6689 - loss: 0.9330 - val_accuracy: 0.6658 - val_loss: 1.2381
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 266s 1s/step - accuracy: 0.6842 - loss: 0.8412 - val_accuracy: 0.5872 - val_loss: 1.0741
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 299s 1s/step - accuracy: 0.7007 - loss: 0.7905 - val_accuracy: 0.6851 - val_loss: 0.8824
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 0.7021 - loss: 0.7708 - val_accuracy: 0.5925 - val_loss: 1.0358
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 276s 1s/step - accuracy: 0.7154 - loss: 0.7520 - val_accuracy: 0.6738 - val_loss: 0.9683


In [100]:
train_loss, hype_train_acc_bn = cnn_bn.evaluate(train_ds)
valid_loss, hype_valid_acc_bn = cnn_bn.evaluate(valid_ds)
test_loss, hype_test_acc_bn = cnn_bn.evaluate(test_ds)
print(hype_train_acc_bn)
print(hype_valid_acc_bn)
print(hype_test_acc_bn)

220/220 ━━━━━━━━━━━━━━━━━━━━ 56s 244ms/step - accuracy: 0.6690 - loss: 0.9480
47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 253ms/step - accuracy: 0.6691 - loss: 0.9648
47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 253ms/step - accuracy: 0.6713 - loss: 0.9765
0.6690441966056824
0.6691078543663025
0.6713240146636963


In [101]:
bn_results.loc[len(bn_results)] = [
    "batch normalization hyperparameter",
    hype_train_acc_bn,
    hype_test_acc_bn,
    hype_valid_acc_bn
]
bn_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,batch normalization using SGD,0.647504,0.640053,0.643808
1,batch normalization using RMSprop,0.610984,0.604125,0.607856
2,batch normalization using adam,0.558060,0.542914,0.551931
3,batch normalization using batchsize 16,0.512696,0.502329,0.517976
4,batch normalization hyperparameter,0.669044,0.671324,0.669108


In [102]:
bn_results.to_csv("bn_cnn_comparison.csv",index=False)

In [103]:
best_bn_cnn.save("cnn_batchnorm_phase5.keras")

In [104]:
cnn_bn.save("cnn_batchnorm_bestmodel.keras")